# SHAP Performance Regimes
This notebook assembles a run-scoped regime analysis table for one XGBoost interpretable-model run, evaluates reduced SHAP representations, compares clustering behavior across performance groups, and inspects the selected cluster SHAP signatures for easy, medium, and hard trajectory outcomes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from data_modelling.prepared_data import load_prepared_data, prepare_single_target_model_data
from data_modelling.run_context import format_exported_model_label, get_exported_model_info, load_run_context
from data_modelling.shap_performance_regimes_utils import (
    assemble_step1_analysis_table,
    build_cluster_shap_profiles,
    evaluate_umap_trustworthiness_by_group,
    format_shap_feature_name,
    get_shap_cols,
    resolve_raw_metric_col,
    run_step2_clustering,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

MODEL_ID = 'xgboost'
RUN_NAME = "nusc_mini_debug_tpp-11_Mar_2026_15_29_02"
EVAL_CSV_NAME = "eval_epoch_5.csv"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'
LOWER_IS_BETTER = True
PERFORMANCE_GROUP_COL = 'performance_group'

CLUSTER_SPEC = {
    'groups': ['easy', 'medium', 'hard'],
    'algorithms': ['hdbscan', 'optics'],
    'evaluate_umap_latent_space': True,
    # The notebook resolves candidate dimensions to 1..(#features - 1) after the SHAP columns are loaded.
    'umap_candidate_dims': None,
    # Set one integer for all groups or a dict like {'easy': 2, 'medium': 4, 'hard': 2} after inspecting the trustworthiness plot.
    'umap_selected_n_components': {'easy': 2, 'medium': 4, 'hard': 2},
    'trustworthiness_n_neighbors': 15,
    'umap_n_neighbors': 15,
    'umap_min_dist': 0.1,
    'random_state': 42,
    'min_cluster_size_fraction': 0.05,
    'min_cluster_size_min': 5,
    'optics_cluster_method': 'xi',
    'optics_xi': 0.05,
    'distance_metric': 'euclidean',
}

INSPECTION_CONFIG = {
    'inspection_algorithm': 'hdbscan',
    'inspection_cluster_space': 'raw',
    'inspection_top_k_features': 8,
    'inspection_top_k_table': 3,
    'sort_cluster_profiles_by': 'cluster_size',
}

if MODEL_ID != 'xgboost':
    raise NotImplementedError("This notebook currently supports MODEL_ID='xgboost' only.")



## Resolve Run Context and Artifact Paths
**Purpose:** Tie every input and output to one exported modelling run and one joined-metrics file.<br>
**Inputs:** `RUN_NAME`, `EVAL_CSV_NAME`, `MODEL_ID`, optional `TARGET_COL`.


In [ ]:
run_ctx = load_run_context(MODEL_ID, RUN_NAME, TARGET_COL)
manifest = run_ctx.manifest
target_col = run_ctx.target_col
feature_cols = run_ctx.feature_cols
exported_model_info = get_exported_model_info(manifest)
exported_model_label = format_exported_model_label(exported_model_info)
raw_metric_col = resolve_raw_metric_col(manifest, target_col)

PREPARED_DATA_PATH = Path('../../results/interpretable_model/prepared_data') / RUN_NAME / f'prepared_data_{raw_metric_col}.csv'
SHAP_VALUES_PATH = run_ctx.tables_dir / f'shap_values_{target_col}.csv'
JOINED_METRICS_PATH = Path('../../results/trajectory_prediction/trajectory_metrics_joined') / RUN_NAME / EVAL_CSV_NAME
RESULTS_ROOT = Path('../../results/interpretable_model/shap_performance_regimes') / RUN_NAME
TABLES_DIR = RESULTS_ROOT / 'tables'
PLOTS_DIR = RESULTS_ROOT / 'plots'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

REGIME_ANALYSIS_PATH = TABLES_DIR / f'regime_analysis_{target_col}.csv'
PERFORMANCE_GROUP_SUMMARY_PATH = TABLES_DIR / f'performance_group_summary_{target_col}.csv'
UMAP_TRUSTWORTHINESS_PATH = TABLES_DIR / f'umap_trustworthiness_{target_col}.csv'
CLUSTER_SCORES_PATH = TABLES_DIR / f'cluster_scores_{target_col}.csv'
CLUSTER_ASSIGNMENTS_PATH = TABLES_DIR / f'cluster_assignments_{target_col}.csv'
CLUSTER_SHAP_PROFILES_PATH = TABLES_DIR / f'cluster_shap_profiles_{target_col}.csv'
UMAP_TRUSTWORTHINESS_PLOT_PATH = PLOTS_DIR / f'umap_trustworthiness_curve_{target_col}.png'
RAW_ALGORITHM_GRID_PATH = PLOTS_DIR / f'raw_algorithm_comparison_grid_{target_col}.png'
UMAP_ALGORITHM_GRID_PATH = PLOTS_DIR / f'umap_algorithm_comparison_grid_{target_col}.png'

required_paths = [
    ('prepared data export', PREPARED_DATA_PATH),
    ('SHAP value export', SHAP_VALUES_PATH),
    ('joined metrics export', JOINED_METRICS_PATH),
]
for label, path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f'Missing required {label}: {path}')

print(f'Run: {RUN_NAME}')
print(f'Eval CSV: {EVAL_CSV_NAME}')
print(f'Exported model: {exported_model_label}')
print(f'Resolved target_col: {target_col}')
print(f'Resolved raw metric col: {raw_metric_col}')
print(f'Feature count: {len(feature_cols)}')
print(f'Prepared data path: {PREPARED_DATA_PATH}')
print(f'SHAP values path: {SHAP_VALUES_PATH}')
print(f'Joined metrics path: {JOINED_METRICS_PATH}')
print(f'Results root: {RESULTS_ROOT.resolve()}')



## Load the Prepared Modelling Table
**Purpose:** Reconstruct the exact modelling rows and feature key used by the interpretable model.


In [ ]:
prepared_df = load_prepared_data(
    PREPARED_DATA_PATH,
    display_fn=display,
    include_missing_summary=True,
    include_dtype_summary=True,
)

prepared = prepare_single_target_model_data(
    prepared_df,
    target_col=target_col,
    default_target=raw_metric_col,
)
model_df = prepared['model_df'].copy()
prepared_feature_cols = prepared['feature_cols']

if prepared['target_col'] != target_col:
    raise ValueError(f"Prepared target mismatch. expected={target_col}, actual={prepared['target_col']}")
if prepared_feature_cols != feature_cols:
    raise ValueError(
        'Prepared feature columns do not match the run manifest exactly. '
        f'expected={feature_cols}, actual={prepared_feature_cols}'
    )

print(f'Prepared modelling rows: {len(model_df)}')
print(f'Prepared feature key size: {len(prepared_feature_cols)}')


## Assemble the Performance-Regime Table
**Purpose:** Join prepared rows, run-scoped metrics, and SHAP exports, then assign `easy` / `medium` / `hard` performance groups.


In [ ]:
joined_metrics_df = pd.read_csv(JOINED_METRICS_PATH)
shap_values_df = pd.read_csv(SHAP_VALUES_PATH)

analysis_df, group_summary_df = assemble_step1_analysis_table(
    prepared_model_df=model_df,
    joined_metrics_df=joined_metrics_df,
    shap_values_df=shap_values_df,
    feature_cols=feature_cols,
    target_col=target_col,
    performance_metric_col=raw_metric_col,
    lower_is_better=LOWER_IS_BETTER,
    performance_group_col=PERFORMANCE_GROUP_COL,
)

shap_cols = get_shap_cols(analysis_df)
CLUSTER_SPEC_RESOLVED = dict(CLUSTER_SPEC)
CLUSTER_SPEC_RESOLVED['umap_candidate_dims'] = list(range(1, len(shap_cols)))

group_counts_df = (
    analysis_df[PERFORMANCE_GROUP_COL]
    .value_counts()
    .rename_axis(PERFORMANCE_GROUP_COL)
    .reset_index(name='count')
)

print(f'Joined metrics rows: {len(joined_metrics_df)}')
print(f'SHAP value rows: {len(shap_values_df)}')
print(f'Analysis rows: {len(analysis_df)}')
print(f'SHAP feature columns available: {len(shap_cols)}')
print(f"UMAP candidate dimensions: {CLUSTER_SPEC_RESOLVED['umap_candidate_dims']}")

display(group_summary_df)
display(group_counts_df)
display(analysis_df.head())


## Evaluate Reduced Representations
**Purpose:** Compute trustworthiness scores for dimensions `1, 2, ..., (#features - 1)` for each performance group, then let the user visually choose the UMAP dimension setting before clustering.


In [ ]:
trustworthiness_df = evaluate_umap_trustworthiness_by_group(
    analysis_df,
    cluster_spec=CLUSTER_SPEC_RESOLVED,
    performance_group_col=PERFORMANCE_GROUP_COL,
    shap_cols=shap_cols,
)

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.lineplot(
    data=trustworthiness_df,
    x='n_components',
    y='trustworthiness',
    hue='performance_group',
    style='performance_group',
    markers=True,
    dashes=False,
    ax=ax,
)
selected_umap_dims = CLUSTER_SPEC_RESOLVED.get('umap_selected_n_components')
if isinstance(selected_umap_dims, int):
    ax.axvline(selected_umap_dims, color='black', linestyle='--', alpha=0.5, label='selected dim')
ax.set_title('UMAP trustworthiness by reduced dimension and performance group')
ax.set_xlabel('Reduced dimension')
ax.set_ylabel('Trustworthiness')
ax.set_xticks(sorted(trustworthiness_df['n_components'].unique().tolist()))
plt.tight_layout()
plt.savefig(UMAP_TRUSTWORTHINESS_PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()

print("Inspect the trustworthiness curve above, then adjust CLUSTER_SPEC['umap_selected_n_components'] if needed before running the clustering cell below.")
display(trustworthiness_df)




## Cluster Within Performance Groups
**Purpose:** Cluster SHAP vectors separately inside each performance group, comparing both algorithms on raw SHAP and the manually selected UMAP latent space when enabled.


In [ ]:
clustering_results = run_step2_clustering(
    analysis_df,
    cluster_spec=CLUSTER_SPEC_RESOLVED,
    performance_group_col=PERFORMANCE_GROUP_COL,
    row_id_col='row_id',
    shap_cols=shap_cols,
)

clustered_df = clustering_results['clustered_df']
cluster_scores_df = clustering_results['cluster_scores_df']
best_cluster_runs_df = cluster_scores_df.loc[cluster_scores_df['selected_for_group']].copy()

inspection_algorithm = INSPECTION_CONFIG['inspection_algorithm']
inspection_cluster_space = INSPECTION_CONFIG['inspection_cluster_space']
if inspection_algorithm not in CLUSTER_SPEC_RESOLVED['algorithms']:
    raise ValueError(
        f"Unsupported inspection_algorithm={inspection_algorithm!r}. "
        f"Expected one of {CLUSTER_SPEC_RESOLVED['algorithms']}"
    )
if inspection_cluster_space not in {'raw', 'umap'}:
    raise ValueError(
        f"Unsupported inspection_cluster_space={inspection_cluster_space!r}. Expected 'raw' or 'umap'."
    )
if inspection_cluster_space == 'umap' and not CLUSTER_SPEC_RESOLVED['evaluate_umap_latent_space']:
    raise ValueError("inspection_cluster_space='umap' requires evaluate_umap_latent_space=True.")

inspected_cluster_runs_df = cluster_scores_df.loc[
    (cluster_scores_df['algorithm'] == inspection_algorithm)
    & (cluster_scores_df['cluster_space'] == inspection_cluster_space)
].copy().sort_values('performance_group')
expected_groups = [group for group in CLUSTER_SPEC_RESOLVED['groups'] if group in set(cluster_scores_df['performance_group'])]
missing_groups = [
    group for group in expected_groups
    if group not in set(inspected_cluster_runs_df['performance_group'])
]
if missing_groups:
    raise ValueError(
        'Inspection selection is missing clustering results for performance groups: '
        f'{missing_groups}. selection=({inspection_algorithm}, {inspection_cluster_space})'
    )

cluster_shap_profiles_df = build_cluster_shap_profiles(
    clustered_df,
    inspected_cluster_runs_df,
    performance_group_col=PERFORMANCE_GROUP_COL,
    shap_cols=shap_cols,
)

print(f'Cluster score rows: {len(cluster_scores_df)}')
print(f'Best-ranked regime runs: {len(best_cluster_runs_df)}')
print(
    'Inspection selection: '
    f"algorithm={inspection_algorithm}, cluster_space={inspection_cluster_space}, "
    f"groups={len(inspected_cluster_runs_df)}"
)
print(f'Cluster profile rows: {len(cluster_shap_profiles_df)}')

display(cluster_scores_df)
display(best_cluster_runs_df)
display(inspected_cluster_runs_df)
display(cluster_shap_profiles_df.head())




## Compare Clustering Outputs
**Purpose:** Show one panel per `(performance group, clustering algorithm)` combination for raw SHAP and reduced SHAP spaces.


In [ ]:
def plot_cluster_comparison_grid(cluster_scores_subset: pd.DataFrame, plot_path: Path, cluster_space_label: str) -> pd.DataFrame:
    groups = CLUSTER_SPEC_RESOLVED['groups']
    algorithms = CLUSTER_SPEC_RESOLVED['algorithms']
    comparison_df = cluster_scores_subset.copy()
    fig, axes = plt.subplots(len(groups), len(algorithms), figsize=(7.2 * len(algorithms), 4.8 * len(groups)), squeeze=False)

    for row_idx, performance_group in enumerate(groups):
        for col_idx, algorithm in enumerate(algorithms):
            ax = axes[row_idx][col_idx]
            selected_rows = comparison_df.loc[
                (comparison_df['performance_group'] == performance_group)
                & (comparison_df['algorithm'] == algorithm)
            ]
            if selected_rows.empty:
                ax.set_visible(False)
                continue

            comparison_row = selected_rows.iloc[0]
            group_df = clustered_df.loc[clustered_df[PERFORMANCE_GROUP_COL] == performance_group].copy()
            label_col = comparison_row['candidate_label_col']
            cluster_ids = [
                int(cluster_id)
                for cluster_id in group_df[label_col].dropna().astype(int).unique().tolist()
            ]
            ordered_cluster_ids = sorted(cluster_id for cluster_id in cluster_ids if cluster_id != -1)
            if -1 in cluster_ids:
                ordered_cluster_ids.append(-1)

            palette = sns.color_palette('tab10', n_colors=max(len(ordered_cluster_ids) - (1 if -1 in ordered_cluster_ids else 0), 1))
            color_lookup = {
                cluster_id: palette[idx % len(palette)]
                for idx, cluster_id in enumerate(cluster_id for cluster_id in ordered_cluster_ids if cluster_id != -1)
            }
            if -1 in ordered_cluster_ids:
                color_lookup[-1] = (0.65, 0.65, 0.65)

            for cluster_id in ordered_cluster_ids:
                cluster_rows = group_df.loc[group_df[label_col].astype('Int64') == cluster_id]
                ax.scatter(
                    cluster_rows['viz_umap_x'],
                    cluster_rows['viz_umap_y'],
                    s=28,
                    alpha=0.88,
                    c=[color_lookup[cluster_id]],
                    edgecolors='none',
                )

            if pd.notna(comparison_row['dbcv']):
                panel_title = f"{algorithm.upper()}\nDBCV={comparison_row['dbcv']:.3f}, clusters={int(comparison_row['n_clusters'])}"
            else:
                panel_title = f"{algorithm.upper()}\nDBCV=NaN, clusters={int(comparison_row['n_clusters'])}"
            ax.set_title(panel_title)
            ax.set_xlabel('UMAP 1')
            if col_idx == 0:
                ax.set_ylabel(f'{performance_group}\nUMAP 2')
            else:
                ax.set_ylabel('')

    fig.suptitle(f'Cluster comparison in {cluster_space_label} space', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    return comparison_df

raw_comparison_df = cluster_scores_df.loc[cluster_scores_df['cluster_space'] == 'raw'].copy()
umap_comparison_df = cluster_scores_df.loc[cluster_scores_df['cluster_space'] == 'umap'].copy()

raw_comparison_df = raw_comparison_df.sort_values(['performance_group', 'algorithm']).reset_index(drop=True)
umap_comparison_df = umap_comparison_df.sort_values(['performance_group', 'algorithm']).reset_index(drop=True)

plot_cluster_comparison_grid(raw_comparison_df, RAW_ALGORITHM_GRID_PATH, 'raw SHAP')
if umap_comparison_df.empty:
    raise ValueError('Reduced-space clustering results are not available. Enable evaluate_umap_latent_space to render the reduced-space comparison grid.')
plot_cluster_comparison_grid(umap_comparison_df, UMAP_ALGORITHM_GRID_PATH, 'reduced SHAP')

print('Raw-space clustering runs:')
display(raw_comparison_df[['performance_group', 'algorithm', 'cluster_space', 'dbcv', 'n_clusters', 'noise_fraction']])
print('Reduced-space clustering runs:')
display(umap_comparison_df[['performance_group', 'algorithm', 'cluster_space', 'dbcv', 'n_clusters', 'noise_fraction']])



## Inspect Regime SHAP Signatures
**Purpose:** Inspect the user-selected clustering variant for each performance group with compact top-driver tables, signed SHAP bar plots, and full cluster-profile heatmaps.


In [ ]:
barplot_manifest_records = []
heatmap_manifest_records = []

sort_key = INSPECTION_CONFIG['sort_cluster_profiles_by']
if sort_key not in {'cluster_size', 'cluster_rank_by_size'}:
    raise ValueError(f'Unsupported sort_cluster_profiles_by value: {sort_key}')

for _, inspected_run in inspected_cluster_runs_df.sort_values('performance_group').iterrows():
    performance_group = inspected_run['performance_group']
    profile_df = cluster_shap_profiles_df.loc[
        cluster_shap_profiles_df['performance_group'] == performance_group
    ].copy()
    if profile_df.empty:
        continue

    ascending_flags = [False, True] if sort_key == 'cluster_size' else [True, True]
    profile_df = profile_df.sort_values([sort_key, 'cluster_id'], ascending=ascending_flags).reset_index(drop=True)
    ordered_shap_cols = (
        profile_df[shap_cols]
        .abs()
        .mean(axis=0)
        .sort_values(ascending=False)
        .index
        .tolist()
    )
    feature_label_lookup = {shap_col: format_shap_feature_name(shap_col) for shap_col in shap_cols}

    metadata_df = pd.DataFrame(
        [
            {
                'performance_group': performance_group,
                'inspection_algorithm': inspected_run['algorithm'],
                'inspection_cluster_space': inspected_run['cluster_space'],
                'dbcv': inspected_run['dbcv'],
                'noise_fraction': inspected_run['noise_fraction'],
                'n_clusters': inspected_run['n_clusters'],
            }
        ]
    )
    print(f'Inspection profile: {performance_group}')
    display(metadata_df)

    top_driver_columns = ['cluster_id', 'cluster_size', 'cluster_size_share']
    for rank in range(1, INSPECTION_CONFIG['inspection_top_k_table'] + 1):
        top_driver_columns.extend([
            f'dominant_feature_{rank}',
            f'dominant_direction_{rank}',
            f'dominant_abs_shap_{rank}',
        ])
    top_driver_df = profile_df[top_driver_columns].copy()
    top_driver_df['cluster_size_share'] = top_driver_df['cluster_size_share'].round(4)
    for rank in range(1, INSPECTION_CONFIG['inspection_top_k_table'] + 1):
        top_driver_df[f'dominant_abs_shap_{rank}'] = top_driver_df[f'dominant_abs_shap_{rank}'].round(4)
    display(top_driver_df)

    n_clusters = len(profile_df)
    fig, axes = plt.subplots(n_clusters, 1, figsize=(10, max(3.2 * n_clusters, 4.5)), squeeze=False)
    for axis_idx, (_, cluster_row) in enumerate(profile_df.iterrows()):
        ax = axes[axis_idx][0]
        top_shap_cols = sorted(
            shap_cols,
            key=lambda shap_col: abs(float(cluster_row[shap_col])),
            reverse=True,
        )[: INSPECTION_CONFIG['inspection_top_k_features']]
        top_shap_cols = list(reversed(top_shap_cols))
        values = [float(cluster_row[shap_col]) for shap_col in top_shap_cols]
        labels = [feature_label_lookup[shap_col] for shap_col in top_shap_cols]
        colors = ['#C44E52' if value > 0 else '#4C72B0' for value in values]
        ax.barh(labels, values, color=colors)
        ax.axvline(0, color='black', linewidth=0.9)
        ax.set_title(
            f"Cluster {int(cluster_row['cluster_id'])} | n={int(cluster_row['cluster_size'])} | share={cluster_row['cluster_size_share']:.2%}"
        )
        ax.set_xlabel('Mean SHAP value')
        ax.set_ylabel('Feature')
    fig.suptitle(
        f"Cluster SHAP profiles - {performance_group} ({inspected_run['algorithm']}, space={inspected_run['cluster_space']})",
        fontsize=15,
        y=1.02,
    )
    plt.tight_layout()
    barplot_path = PLOTS_DIR / f'inspected_cluster_profile_barplot_{performance_group}_{target_col}.png'
    plt.savefig(barplot_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    heatmap_index = [
        f"Cluster {int(cluster_id)} (n={int(cluster_size)}, share={cluster_share:.1%})"
        for cluster_id, cluster_size, cluster_share in zip(
            profile_df['cluster_id'],
            profile_df['cluster_size'],
            profile_df['cluster_size_share'],
        )
    ]
    heatmap_plot_df = profile_df[ordered_shap_cols].copy()
    heatmap_plot_df.columns = [feature_label_lookup[shap_col] for shap_col in ordered_shap_cols]
    heatmap_plot_df.index = pd.Index(heatmap_index)
    heatmap_path = PLOTS_DIR / f'inspected_cluster_profile_heatmap_{performance_group}_{target_col}.png'

    fig, ax = plt.subplots(figsize=(max(10, len(ordered_shap_cols) * 0.55), max(4.5, len(profile_df) * 0.9)))
    sns.heatmap(
        heatmap_plot_df,
        cmap='coolwarm',
        center=0,
        ax=ax,
    )
    ax.set_title(
        f"Cluster profile heatmap - {performance_group} ({inspected_run['algorithm']}, space={inspected_run['cluster_space']})"
    )
    ax.set_xlabel('Feature')
    ax.set_ylabel('Cluster')
    plt.tight_layout()
    plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    barplot_manifest_records.append(
        {
            'performance_group': performance_group,
            'inspection_algorithm': inspected_run['algorithm'],
            'inspection_cluster_space': inspected_run['cluster_space'],
            'barplot_plot_path': str(barplot_path),
        }
    )
    heatmap_manifest_records.append(
        {
            'performance_group': performance_group,
            'inspection_algorithm': inspected_run['algorithm'],
            'inspection_cluster_space': inspected_run['cluster_space'],
            'heatmap_plot_path': str(heatmap_path),
        }
    )

barplot_manifest_df = pd.DataFrame(barplot_manifest_records)
heatmap_manifest_df = pd.DataFrame(heatmap_manifest_records)
display(barplot_manifest_df)
display(heatmap_manifest_df)



## Export Regime Artifacts
**Purpose:** Persist the assembled regime table, clustering outputs, and selected cluster-profile plots for downstream interpretation.


In [ ]:
analysis_df.to_csv(REGIME_ANALYSIS_PATH, index=False)
group_summary_df.to_csv(PERFORMANCE_GROUP_SUMMARY_PATH, index=False)
trustworthiness_df.to_csv(UMAP_TRUSTWORTHINESS_PATH, index=False)
cluster_scores_df.to_csv(CLUSTER_SCORES_PATH, index=False)
clustered_df.to_csv(CLUSTER_ASSIGNMENTS_PATH, index=False)
cluster_shap_profiles_df.to_csv(CLUSTER_SHAP_PROFILES_PATH, index=False)

plot_manifest_df = pd.concat(
    [
        pd.DataFrame(
            [
                {'plot_type': 'umap_trustworthiness_curve', 'path': str(UMAP_TRUSTWORTHINESS_PLOT_PATH)},
                {'plot_type': 'raw_algorithm_comparison_grid', 'path': str(RAW_ALGORITHM_GRID_PATH)},
                {'plot_type': 'umap_algorithm_comparison_grid', 'path': str(UMAP_ALGORITHM_GRID_PATH)},
            ]
        ),
        barplot_manifest_df.rename(columns={'barplot_plot_path': 'path'}).assign(plot_type='inspected_cluster_profile_barplot')[['plot_type', 'performance_group', 'path']],
        heatmap_manifest_df.rename(columns={'heatmap_plot_path': 'path'}).assign(plot_type='inspected_cluster_profile_heatmap')[['plot_type', 'performance_group', 'path']],
    ],
    ignore_index=True,
    sort=False,
)

print('Saved artifacts:')
print(f'- Run manifest:               {run_ctx.manifest_path}')
print(f'- Prepared data export:       {PREPARED_DATA_PATH}')
print(f'- SHAP value export:          {SHAP_VALUES_PATH}')
print(f'- Joined metrics export:      {JOINED_METRICS_PATH}')
print(f'- Regime analysis table:      {REGIME_ANALYSIS_PATH}')
print(f'- Performance group summary:  {PERFORMANCE_GROUP_SUMMARY_PATH}')
print(f'- UMAP trustworthiness:       {UMAP_TRUSTWORTHINESS_PATH}')
print(f'- Trustworthiness curve:      {UMAP_TRUSTWORTHINESS_PLOT_PATH}')
print(f'- Cluster scores:             {CLUSTER_SCORES_PATH}')
print(f'- Cluster assignments:        {CLUSTER_ASSIGNMENTS_PATH}')
print(f'- Cluster SHAP profiles:      {CLUSTER_SHAP_PROFILES_PATH}')
print(
    '- Inspection selection:      '
    f"algorithm={INSPECTION_CONFIG['inspection_algorithm']}, "
    f"space={INSPECTION_CONFIG['inspection_cluster_space']}"
)
print(f'- Raw comparison grid:        {RAW_ALGORITHM_GRID_PATH}')
print(f'- Reduced comparison grid:    {UMAP_ALGORITHM_GRID_PATH}')
print(f'- Plot directory:             {PLOTS_DIR}')

display(plot_manifest_df)

